# Session1_Task9 — Customer Lifetime Value (CLTV)

## สูตร CLTV
`CLTV = Average Purchase Value × Purchase Frequency × 36`
- **Average Purchase Value** = รายได้รวม ÷ จำนวน transactions ของลูกค้าคนนั้น
- **Purchase Frequency** = จำนวน transactions ÷ จำนวนเดือนที่มีข้อมูล
- **36** = ระยะเวลา 36 เดือน (3 ปี)

In [ ]:
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

s = pd.read_csv('sales_transactions_cleaned.csv')
s['date']    = pd.to_datetime(s['date'])
s['revenue'] = (s['quantity'] * s['price']) - pd.to_numeric(s['discount_amount'], errors='coerce').fillna(0)

In [ ]:
# --- คำนวณ Average Purchase Value ต่อลูกค้า ---

# total revenue และ transaction count ต่อลูกค้า
cust = s.groupby('customer_id').agg(
    total_revenue=('revenue',        'sum'),
    total_tx     =('transaction_id', 'nunique')  # nunique → นับไม่ซ้ำ
).reset_index()

# avg_purchase_value = total_revenue / total_transactions
cust['avg_purchase_value'] = cust['total_revenue'] / cust['total_tx']
print('avg_purchase_value sample:')
display(cust.head())

In [ ]:
# --- คำนวณ Purchase Frequency (transactions per month) ---

# จำนวนเดือนที่มีข้อมูลทั้งหมดในไฟล์
# .dt.to_period('M').nunique() → นับจำนวนเดือนที่ไม่ซ้ำ
total_months = s['date'].dt.to_period('M').nunique()
print(f'Total months in data: {total_months}')

# purchase_frequency = transactions ÷ จำนวนเดือนทั้งหมด
cust['purchase_frequency'] = cust['total_tx'] / total_months

# --- คำนวณ CLTV ---
# CLTV = avg_purchase_value × purchase_frequency × 36
# .round(2) → ปัดทศนิยม 2 ตำแหน่ง
cust['cltv'] = (cust['avg_purchase_value'] * cust['purchase_frequency'] * 36).round(2)

result = cust[['customer_id','cltv']].copy()
result['customer_id'] = result['customer_id'].astype(int)

display(result.head())
print(f'\nCLTV range: {result["cltv"].min():.2f} – {result["cltv"].max():.2f}')

result.to_csv('Session1_CLTV.csv', index=False)
print('✅ Saved Session1_CLTV.csv')

# === จุดสังเกต ===
# ✔ ไฟล์มี 2 คอลัมน์: customer_id (int), cltv (float)
# ✔ cltv ทุกค่าเป็นบวก
# ✔ จำนวนแถว = จำนวน unique customer ในไฟล์ sales